[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Image_Processing.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Image Processing as 2-D DSP

Images are just signals with two time axes. Everything from [Foundations](./Foundations_of_Signal_Processing_1.ipynb) transfers — convolution, spectra, sampling — and the payoff is a straight bridge into [CNNs](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb).

## 1. Pre-requisites

[Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) S5 (DFT & sampling), [Filter Design](./Filter_Design.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

# Synthesize a test image (no downloads, fully reproducible):
# a "room": gradient wall, striped rug, circular table, textured window
yy, xx = np.mgrid[0:512, 0:512] / 512
img = 0.35 + 0.3 * yy                                        # lit wall
img += 0.25 * ((np.sin(2*np.pi*18*xx) > 0) & (yy > 0.72))    # striped rug
img += 0.3 * (((xx-0.68)**2 + (yy-0.42)**2) < 0.03)          # table disc
win = (np.abs(xx-0.22) < 0.13) & (np.abs(yy-0.28) < 0.16)
img += win * (0.25 + 0.12 * np.sin(2*np.pi*40*(xx+yy)))      # textured window
img = np.clip(img + 0.02 * rng.standard_normal((512, 512)), 0, 1)

plt.figure(figsize=(3.4, 3.4)); plt.imshow(img, cmap="gray"); plt.axis("off")
plt.title("our (synthetic) test subject"); plt.tight_layout(); plt.show()

**What just happened.** A synthetic room, built from four ingredients chosen so that each exercises a different part of the workshop. The gradient wall is a very low spatial frequency. The striped rug is a strong single frequency in one orientation. The table disc is a smooth region bounded by a curved edge — energy at every orientation. The window texture is a high-frequency diagonal grating, and the added noise gives the high frequencies a floor.

Everything is synthesised rather than downloaded, which matters more than convenience: the ground truth is known exactly. When the spectrum shows a bright pair of points, we can say *which* line of code put them there, and when a filter removes something we can name what was lost. Photographs make prettier demos and worse experiments.

**The one idea to hold for the rest of the workshop.** An image is a signal with two axes rather than one. Every tool from [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) transfers by adding an index: convolution becomes a kernel sliding in two directions, the DFT decomposes into 2-D plane waves instead of sinusoids, and Nyquist applies per axis. Nothing conceptually new is required — which is precisely why a DSP background is such a strong entry into computer vision.

Keep an eye on the rug and the window in particular. They are the highest-frequency structures present, so they will be the first things the blur destroys in Session 1 and the first to alias in Session 2.

---
### 🕐 Session 1 of 3 — *2-D Convolution & the 2-D Spectrum* (~35 min)
**Goal:** extend filtering and Fourier to two dimensions; learn to read a 2-D spectrum.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb). &nbsp; **Feeds into:** Session 2 (sampling & aliasing).

---

## 2. Two Axes, Same Rules

💡 **Intuition.** 2-D convolution slides a small *kernel* over the image — identical mechanics to 1-D, once per axis. The 2-D DFT decomposes an image into **plane waves**: gratings of every orientation and spatial frequency. Reading the spectrum: center = DC (average brightness), distance from center = fineness of detail, *direction* = orientation of the stripes producing it. Edges in direction θ light up spectrum energy perpendicular to θ.

In [ ]:
# Blur = 2-D low-pass; the spectrum shows exactly what was removed

# YOUR CODE HERE


**What just happened.** Four panels: the image, its spectrum, the blurred image, and the blurred spectrum. The pair of spectra is the informative comparison — the original has energy spread well out toward the corners, and after blurring the outer regions have gone dark. Low-pass filtering, and the spectrum showing exactly what was removed.

**Three rules for reading a 2-D spectrum, all visible here.** The centre is DC, the mean brightness — always the brightest point. Distance from centre is *fineness* of detail, so the outer regions are fine texture. And direction from centre gives the orientation of the grating responsible — with the important twist that it is **perpendicular** to the edges you see in the image. The rug's vertical stripes therefore deposit their energy along the *horizontal* axis of the spectrum, which catches nearly everyone out the first time.

Match each feature to its signature. The rug is a single frequency at a single orientation, so it appears as a compact pair of bright points placed symmetrically about the centre. The window texture is a diagonal grating, so its points sit on the diagonal, further out because the frequency is higher. The table disc is a *curved* edge containing every orientation, so it contributes a diffuse radial spread rather than isolated spots. And the wall gradient sits almost on top of DC.

**Compare the two images and check the prediction.** The blur destroyed the window texture and softened the rug stripes — exactly the two highest-frequency features, exactly as the spectra say. The wall and the table survive nearly untouched because they were low-frequency to begin with. "Blur removes detail" is the informal statement; "convolution with a smooth kernel attenuates high spatial frequencies" is the same claim with a mechanism, and the second one predicts *which* details will go.

**Two implementation details worth noticing.** The kernel is `np.outer(np.hanning(15), np.hanning(15))` — a **separable** filter, the outer product of two 1-D windows. That means it can be applied as two 1-D passes: 30 multiplies per pixel instead of 225, a 7.5× saving that scales worse and worse in the naive version as kernels grow. Separability is why most production image filters look the way they do, and why depthwise-separable convolutions exist in efficient CNNs.

And `boundary="symm"` mirrors the image beyond its edges. The alternative, zero-padding, invents a black border, and any subsequent edge detector will faithfully report that fictional border as a strong edge. Choosing a boundary condition is not optional in 2-D — you are always making one assumption or another about what lies outside the frame.

---
### 🕐 Session 2 of 3 — *Sampling, Aliasing & Moiré* (~35 min)
**Goal:** see 2-D aliasing with your own eyes; fix it the honest way.
**Builds on:** Session 1; [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) S2. &nbsp; **Feeds into:** Session 3 (edges).

---

## 3. Aliasing You Can See

💡 **Intuition.** Downsampling an image without low-passing folds fine patterns into coarse fake ones — **moiré**. It's the wagon-wheel effect with a second axis: the shirt stripes on TV that swim with rainbow bands are spatial frequencies beyond the sensor's Nyquist, aliased into visibility. The cure is the same as [1-D decimation](./Foundations_of_Signal_Processing_2.ipynb): blur (anti-alias filter) *before* subsampling — cameras do it optically.

In [ ]:
# A radial chirp: frequency grows outward — the classic aliasing stress test

# YOUR CODE HERE


**What just happened.** Aliasing, made visible. The zone plate on the left sweeps smoothly outward through every spatial frequency. Decimate it naively by 4 (middle) and the outer region fills with **rings that were never in the original**. Blur before decimating (right) and the fake rings vanish, leaving an image that is softer but honest.

**And the failure radius is predictable, not just observable.** With $x, y \in [-1,1]$ across 512 pixels, the local radial frequency of $\cos(2\pi \cdot 48 r^2)$ works out to $0.375r$ cycles per pixel. At full resolution Nyquist is 0.5, so the original aliases nowhere — it is honestly sampled everywhere. After decimating by 4 the effective Nyquist falls to $0.125$, so aliasing must begin at

$$r = \frac{0.125}{0.375} = \frac{1}{3}.$$

Look at the middle panel: the inner third of the radius is clean, and the corruption starts right where the arithmetic says. This is not a qualitative observation about a picture — it is a quantitative prediction the image confirms.

**Why aliasing is worse than blur, which is the point of the session.** Blur destroys information and *looks* like it has: a blurry image is obviously blurry, and nobody is fooled. Aliasing produces high-contrast, confident, detailed structure that is entirely fictitious. Those rings would survive an edge detector, a texture classifier, or a neural network without complaint, because nothing about them signals that they are artifacts. **Aliased data does not look damaged, it looks wrong.** That asymmetry is why the anti-aliasing filter is not optional.

**The fix costs real information, and that is acceptable.** The right-hand panel genuinely lost the fine outer detail — it has *less* information than the naive version, in a strict sense. It is still the correct answer, because the detail the naive version appears to retain was never representable at four-fold decimation; what it shows there is fabricated. Choosing honest loss over dishonest detail is the whole judgement.

This is the same rule as [1-D decimation](./Foundations_of_Signal_Processing_2.ipynb): filter, then downsample, never the other way. Cameras implement it optically, with an anti-aliasing filter in front of the sensor that blurs very slightly on purpose — and some manufacturers have removed it to win sharpness, accepting visible moiré on fabrics as the price. The theory in this cell is a product decision somebody actually had to make.

---
### 🕐 Session 3 of 3 — *Edges → Learned Features* (~40 min)
**Goal:** hand-designed gradient kernels, then the straight line to CNNs.
**Builds on:** Session 2.

---

## 4. Edge Detection

💡 **Intuition.** An edge is a spatial derivative — and differentiation is a high-pass filter. The Sobel kernels estimate the gradient along each axis while smoothing along the other; magnitude gives edge strength, arctangent gives orientation. Every classical vision pipeline (and the *first layer of every trained CNN*, as you saw in the [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb)) starts with filters of exactly this shape — the difference is that CNNs *learn* theirs.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three views of the same gradient computation. Magnitude (left) marks *where* edges are — the table's rim, the rug's stripes, the window frame — on a background that is essentially black, because smooth regions have no gradient. Orientation (middle) colours each strong edge by *which way it faces*. Thresholding (right) turns a continuous strength map into a binary decision.

**The kernel is not magic; it is a regularised derivative.** Read `sobel_x` column by column: $[-1, 0, 1]$ horizontally is a finite difference, an approximation to $\partial/\partial x$. Read it row by row: $[1, 2, 1]$ vertically is a smoothing weight. Sobel differentiates along one axis while *averaging* along the perpendicular one, and the averaging is essential — differentiation is a high-pass operation and noise lives at high frequencies, so a bare difference operator is a noise amplifier. This is the same resolution-versus-noise tension that runs through the whole curriculum, resolved here by building the smoothing into the kernel.

**The orientation panel deserves more attention than it usually gets.** Magnitude answers "is there an edge?"; $\arctan2(g_y, g_x)$ answers "which way does it point?" Look at the table's rim: the hue rotates continuously around the circle as the edge direction turns, while the rug's parallel stripes all share one hue. That extra information is what makes edges *composable* — two edges meeting at different orientations is a corner, a consistent set of orientations traces a contour. It is why HOG and SIFT descriptors are built from *oriented* gradients rather than magnitudes, and why orientation is the useful signal for shape rather than brightness alone.

**And the threshold is a decision, not a computation.** `np.quantile(mag, 0.93)` declares the strongest 7% of pixels to be edges — a choice with no principled basis in the data. Too low and texture and noise become "edges"; too high and genuine weak edges vanish. Classical pipelines like Canny add machinery around this (hysteresis thresholding, non-maximum suppression) precisely because a single global threshold is inadequate on real images. Worth noticing that the hardest part of classical edge detection is not the derivative — it is deciding what counts.

**Which is the bridge to the next workshop.** The first layer of a trained [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb), when visualised, contains oriented edge detectors that look strikingly like these kernels. Nobody designed them; gradient descent found them, because natural images are dominated by edges and oriented derivatives are simply what an efficient first representation of such images looks like. Two completely different routes — a century of human engineering, and a loss function — arrive at the same filters. That convergence is about the strongest evidence available that the representation is right.

In [ ]:
# Unsharp masking: the 100-year-old trick still inside every photo app

# YOUR CODE HERE


**What just happened.** The right panel looks crisper — and no detail was added, because none could be. Sharpening cannot invent information the image never contained; all it does is *amplify what is already there*.

**Read the algebra and the name stops being mysterious.** With $\lambda = 1.2$:

$$\text{sharpened} = \text{img} + \lambda(\text{img} - \text{blurred}) = (1+\lambda)\,\text{img} - \lambda\,\text{blurred}.$$

The bracket $(\text{img} - \text{blurred})$ is the original minus its low-pass component — which is exactly a **high-pass** image, the detail the blur removed in Session 1. So "sharpening" is adding back an amplified copy of the high frequencies. The oddly-named "unsharp mask" is literally that: a mask built from an *unsharp* (blurred) copy, subtracted. The technique predates digital imaging entirely — it was done in darkrooms by sandwiching a negative with a blurred positive — and the same three lines are in every photo app on every phone.

**The consequences of pushing $\lambda$ are predictable from the formula.** Raise it and two things happen together, because the high-pass image contains both. Edges gain bright and dark halos, since amplifying the difference overshoots on both sides of a step — this is Gibbs-like ringing, and it is the tell-tale look of an over-processed photograph. And *noise is amplified along with detail*, because noise is high-frequency too, and nothing in the operation distinguishes wanted high frequencies from unwanted ones. Sharpening is a filter, not an understanding.

That limitation is worth stating plainly against the rest of the workshop. Session 2's anti-aliasing filter throws information away and is *correct* to do so. Sharpening amplifies what remains and cannot recover what Session 2 discarded. A blurred image has genuinely lost its high frequencies; multiplying the near-zero residue does not bring them back, it only magnifies the noise sitting where they used to be.

**Pulling the workshop together.** Blur is low-pass, sharpening is high-pass boost, edges are derivatives, moiré is aliasing, and the spectrum shows you what each one did. It is the same theory as [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb), with a second index — which is precisely why a DSP background makes computer vision feel familiar rather than foreign.

## 5. Conclusion

Same theory, second axis: plane-wave spectra, moiré as visible aliasing, edges as gradients, sharpening as boosted high-pass. When the [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) learns its first layer, it rediscovers this session.

---
## Where next

- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — kernels chosen by gradient descent instead of by Sobel.
- [Compressed Sensing](./Compressed_Sensing.ipynb) — images from far fewer samples than pixels.